In [1]:
import torch
from torch import nn
import itertools
import tqdm
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import random
import glob
from torchvision import transforms
from PIL import Image
import os
import numpy as np
import sys


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}") 

DATA_ROOT = '/kaggle/input/datasets/shubham1921/real-to-ghibli-image-dataset-5k-paired-images'
TRAIN_A = os.path.join(DATA_ROOT, 'trainA')
TRAIN_B = os.path.join(DATA_ROOT, 'trainB')
SAVE_DIR = '/kaggle/working/'

Running on: cuda


In [2]:
class Upsample_block(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.upsample = nn.ConvTranspose2d(in_ch, out_ch, 3, stride=2, padding=1, output_padding=1)
        self.norm = nn.InstanceNorm2d(out_ch)
        self.dropout = nn.Dropout(0.5)
        self.activation = nn.GELU()

    def forward(self, X):
        return self.activation(self.dropout(self.norm(self.upsample(X))))


class Conv_block(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=2, \
                 use_leaky=True, use_init_norm=True, use_pad=True):
        super().__init__()
        if use_pad:
            self.conv_layer = nn.Conv2d(in_ch, out_ch, kernel_size, stride, 1, bias=True)
        else:
            self.conv_layer = nn.Conv2d(in_ch, out_ch, kernel_size, stride, 0, bias=True)

        if use_leaky:
            self.activation = nn.LeakyReLU(negative_slope=0.2, inplace=True)
        else:
            self.activation = nn.GELU()

        if use_init_norm:
            self.norm = nn.InstanceNorm2d(out_ch)
        else:
            self.norm = nn.BatchNorm2d(out_ch)

    def forward(self, X):
        return self.activation(self.norm(self.conv_layer(X)))


class Res_Block(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.pad = nn.ReflectionPad2d(1)
        self.conv_block = Conv_block(ch, ch, 3, 1, False, True, False)
        self.conv = nn.Conv2d(ch, ch, 3, 1, 0, bias=True)
        self.dropout = nn.Dropout(0.5)
        self.norm = nn.InstanceNorm2d(ch)

    def forward(self, X):
        X1 = self.dropout(self.conv_block(self.pad(X)))
        X2 = self.norm(self.conv(self.pad(X1)))
        return X2 + X

In [3]:
class Generator(nn.Module):
    def __init__(self, in_ch, out_ch, ResBlock_num=6):
        super().__init__()
        
        # Initial Convolution
        model = [
            nn.ReflectionPad2d(3),
            Conv_block(in_ch, 64, 7, 1, False, True, False),
            Conv_block(64, 128, 3, 2, False, True, True),
            Conv_block(128, 256, 3, 2, False, True, True)
        ]
        
        # Feature Extraction (Residual Blocks)
        for _ in range(ResBlock_num):
            model.append(Res_Block(256))
        
        # Upsampling
        model.append(Upsample_block(256, 128))
        model.append(Upsample_block(128, 64))
        
        # Global Feature Extraction & Output
        model.append(nn.ReflectionPad2d(3))
        model.append(nn.Conv2d(64, out_ch, 7, stride=1, padding=0))
        model.append(nn.Tanh())
        
        self.model = nn.Sequential(*model)

    def forward(self, X):
        return self.model(X)


class Discriminator(nn.Module):
    def __init__(self, in_ch, num_layers=4):
        super().__init__()
        
        layers = [
            nn.Conv2d(in_ch, 64, 4, stride=2, padding=1),
            nn.LeakyReLU(negative_slope=0.2, inplace=True),
        ]
        
        # Layer 1: 64 -> 128
        layers.append(Conv_block(64, 128, 4, 2, True, True, True))
        
        # Layer 2: 128 -> 256
        layers.append(Conv_block(128, 256, 4, 2, True, True, True))
        
        
        # Layer 3: 256 -> 512
        layers.append(Conv_block(256, 512, 4, 1, True, True, True))
        
        # Final Layer
        layers.append(nn.Conv2d(512, 1, 4, stride=1, padding=1))
        
        self.model = nn.Sequential(*layers)

    def forward(self, X):
        return self.model(X)

In [4]:
def init_weight(m):
    class_name = m.__class__.__name__
    if hasattr(m, "weight") and ("Conv" in class_name or "Linear" in class_name):
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if hasattr(m, "bias") and m.bias is not None:
            nn.init.zeros_(m.bias.data)
    elif "BatchNorm2d" in class_name:
        nn.init.normal_(m.weight, 1.0, std=0.02)
        nn.init.zeros_(m.bias)

def Frozen(nets, training=True):
    for net in nets:
        for param in net.parameters():
            param.requires_grad = training

def seed_everything(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.benchmark = True

def reverse_img(img,mean=[0.5]*3,std=[0.5]*3):
    for img_channel,mean_channel,std_channel in zip(img,mean,std):
        img_channel.mul_(std_channel).add_(mean_channel)
    return img.clamp(0,1)

class sample_fake(object):
    def __init__(self, max_imgs=50):
        self.max_imgs = max_imgs
        self.imgs = []

    def __call__(self, imgs):
        ret = []
        for img in imgs:
            if len(self.imgs) < self.max_imgs:
                self.imgs.append(img)
                ret.append(img)
            else:
                if np.random.rand() > 0.5:
                    idx = np.random.randint(0, self.max_imgs)
                    ret.append(self.imgs[idx])
                    self.imgs[idx] = img
                else:
                    ret.append(img)
        return torch.stack(ret)

class lr_sched():
    def __init__(self, decay_epochs=100, total_epochs=200):
        self.decay_epochs = decay_epochs
        self.total_epochs = total_epochs

    def step(self, epoch_num):
        if epoch_num <= self.decay_epochs:
            return 1.0
        else:
            fract = (epoch_num - self.decay_epochs)  / (self.total_epochs - self.decay_epochs)
            return max(0 , 1.0 - fract)

class Accumulator:
    def __init__(self,num):
        self.metric=[0]*num
    def __getitem__(self,idx):
        return self.metric[idx]
    def add(self,*args):
        if args:
            for i,arg in enumerate(args):
                self.metric[i]+=arg

class Recorder:
    def __init__(self,num):
        self.metric=[[] for _ in range(num)]
    def __getitem__(self,idx):
        return self.metric[idx]
    def __len__(self):
        return len(self.metric)
    def add(self,*args):
        if args:
            for i,arg in enumerate(args):
                self.metric[i].append(arg)

In [5]:
class CycleGAN:
    def __init__(self, in_ch, out_ch, configue, show_iter=None):
        self.configue = configue
        self.epochs = configue["epochs"]
        self.start_lr = configue["start_lr"]
        self.temp = configue["temp"]
        self.decay_epoch = configue["decay_epoch"] if configue["decay_epoch"] else int(self.epochs / 2)
        self.idt_coef = configue["idt_coef"]
        self.device = configue["device"]
        self.show_iter = show_iter
        
        # Instantiate Models
        self.G_src2Photo = Generator(in_ch, out_ch)
        self.G_Photo2src = Generator(in_ch, out_ch)
        self.D_src = Discriminator(in_ch)
        self.D_Photo = Discriminator(in_ch)

        self.mseloss = nn.MSELoss()
        self.l1loss = nn.L1Loss()
        
        # Optimizers
        self.G_adam = torch.optim.Adam(
            itertools.chain(self.G_src2Photo.parameters(), self.G_Photo2src.parameters()),
            lr=self.start_lr, betas=(0.5, 0.999))
        self.D_adam = torch.optim.Adam(
            itertools.chain(self.D_src.parameters(), self.D_Photo.parameters()),
            lr=self.start_lr, betas=(0.5, 0.999))

        self.sample_src = sample_fake()
        self.sample_photo = sample_fake()

        # Schedulers
        G_lr = lr_sched(self.decay_epoch, self.epochs)
        D_lr = lr_sched(self.decay_epoch, self.epochs)
        self.G_sched = torch.optim.lr_scheduler.LambdaLR(self.G_adam, G_lr.step)
        self.D_sched = torch.optim.lr_scheduler.LambdaLR(self.D_adam, D_lr.step)
        self.record_metric = Recorder(2)
        
        self.init_model()

    def init_model(self):
        self.G_src2Photo.apply(init_weight)
        self.G_Photo2src.apply(init_weight)
        self.D_src.apply(init_weight)
        self.D_Photo.apply(init_weight)

        self.G_src2Photo = self.G_src2Photo.to(self.device)
        self.G_Photo2src = self.G_Photo2src.to(self.device)
        self.D_src = self.D_src.to(self.device)
        self.D_Photo = self.D_Photo.to(self.device)

        # Optimization for PyTorch 2.0+
        if hasattr(torch, "compile"):
            print("Compiling models with torch.compile...")
            try:
                self.G_src2Photo = torch.compile(self.G_src2Photo)
                self.G_Photo2src = torch.compile(self.G_Photo2src)
                self.D_src = torch.compile(self.D_src)
                self.D_Photo = torch.compile(self.D_Photo)
            except Exception as e:
                print(f"Compilation failed: {e}")

    def visualize_result(self):
        """Helper to visualize progress using the stored iterator."""
        self.G_Photo2src.eval()
        if self.show_iter:
            try:
                try:
                    src_img, _ = next(self.show_iter)
                except StopIteration:
                    return
                
                img = src_img.to(self.device)
                with torch.no_grad():
                    fake = self.G_Photo2src(img)

                fake_img = reverse_img(fake.cpu()[0]).permute(1, 2, 0).numpy()
                real_img = reverse_img(img.cpu()[0]).permute(1, 2, 0).numpy()

                plt.figure(figsize=(10, 5))
                plt.subplot(1, 2, 1)
                plt.title("Original Source")
                plt.imshow(real_img)
                plt.axis("off")

                plt.subplot(1, 2, 2)
                plt.title("Generated Source")
                plt.imshow(fake_img)
                plt.axis("off")
                plt.show()
                plt.close()
            except Exception as e:
                print(f"Visualization failed: {e}")
        self.G_Photo2src.train()

In [6]:
class ImageDataset(Dataset):
    def __init__(self, src_path, photo_path, size=(256,256)):
        super().__init__()
        # Ensure only images are picked up
        self.src_path = glob.glob(src_path + "/*.*")
        self.photo_path = glob.glob(photo_path + "/*.*")
        
        # Simple filter for images
        self.src_path = [x for x in self.src_path if x.lower().endswith(('.png', '.jpg', '.jpeg'))]
        self.photo_path = [x for x in self.photo_path if x.lower().endswith(('.png', '.jpg', '.jpeg'))]

        self.transforms = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize(286),
            transforms.RandomCrop(size),
            transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
        ])

    def __len__(self):
        return min(len(self.src_path), len(self.photo_path))

    def __getitem__(self, idx):
        photo_idx = random.randint(0, len(self.photo_path)-1)
        src_img = self.transforms(Image.open(self.src_path[idx]).convert("RGB"))
        photo_img = self.transforms(Image.open(self.photo_path[photo_idx]).convert("RGB"))
        return src_img, photo_img

# Configuration
configue = {
    "batch_size": 4,
    "seed": 69,
    "epochs": 50,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "start_lr": 2e-4,
    "temp": 10,
    "idt_coef": 0.5,
    "decay_epoch": 15,
    "save": True,
    "training": True
}

def show_configue():
    print("====== Configuration ======")
    for key,value in configue.items():
        print(f"{key:20} {value}")
    print("\n")

In [7]:
SRC_PATH = "/kaggle/input/datasets/shubham1921/real-to-ghibli-image-dataset-5k-paired-images/dataset/trainA"
PHOTO_PATH = "/kaggle/input/datasets/shubham1921/real-to-ghibli-image-dataset-5k-paired-images/dataset/trainB_ghibli"

if not os.path.exists(SRC_PATH):
    os.makedirs(SRC_PATH, exist_ok=True)
    print(f"Created placeholder directory: {SRC_PATH}. Please add images.")
if not os.path.exists(PHOTO_PATH):
    os.makedirs(PHOTO_PATH, exist_ok=True)
    print(f"Created placeholder directory: {PHOTO_PATH}. Please add images.")

seed_everything(configue["seed"])

# 1. Prepare Data
dataset = ImageDataset(SRC_PATH, PHOTO_PATH)

if len(dataset) == 0:
    print("No images found in dataset folders. Exiting.")
else:
    dataloader = DataLoader(
        dataset, 
        batch_size=configue["batch_size"], 
        shuffle=True, 
        pin_memory=True,
        num_workers=2
    )

    # 2. Initialize Model
    cycle_gan = CycleGAN(
        in_ch=3, 
        out_ch=3, 
        configue=configue, 
        show_iter=iter(dataloader)
    )

    # 3. Start Training
    print("Starting training...")
    show_configue()

Compiling models with torch.compile...
Starting training...
====== Configuration ======
batch_size           4
seed                 69
epochs               50
device               cuda
start_lr             0.0002
temp                 10
idt_coef             0.5
decay_epoch          15
save                 True
training             True




In [8]:
import torch
import os
import tqdm
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

def toggle_grad(models, requires_grad):
    """Enables or disables gradient calculation for a list of models."""
    for model in models:
        for param in model.parameters():
            param.requires_grad = requires_grad

# --- Helper: Reverse Normalization ---
def denormalize(img):
    """Converts a normalized tensor (-1 to 1) back to (0 to 1) for visualization."""
    return img * 0.5 + 0.5

# --- Helper: Visualization ---
def visualize_grid(cycle_gan, dataloader, epoch, device):
    """Generates and plots a 2xN grid: [Real A, Fake B] / [Real B, Fake A]"""
    cycle_gan.G_src2Photo.eval()
    cycle_gan.G_Photo2src.eval()
    
    try:
        src_img, photo_img = next(iter(dataloader))
    except StopIteration:
        return

    src_img = src_img.to(device)
    photo_img = photo_img.to(device)

    with torch.no_grad():
        fake_photo = cycle_gan.G_src2Photo(src_img)
        fake_src = cycle_gan.G_Photo2src(photo_img)

    # Denormalize
    real_A = denormalize(src_img.cpu())
    fake_B = denormalize(fake_photo.cpu())
    real_B = denormalize(photo_img.cpu())
    fake_A = denormalize(fake_src.cpu())

    # Create strips: Source Domain on Top, Target Domain on Bottom
    # Top Row: Real Source -> Generated Photo
    row1 = torch.cat((real_A, fake_B), dim=3) # Concatenate width-wise
    # Bottom Row: Real Photo -> Generated Source
    row2 = torch.cat((real_B, fake_A), dim=3) 
    
    # Grid: Stack rows vertically
    grid_tensor = torch.cat((row1, row2), dim=2) 
    
    # Make a grid from the batch (using the first 4 images to keep it readable)
    # If batch size is 1, this just works as is.
    display_grid = make_grid(grid_tensor[:4], nrow=1, padding=5, normalize=False)

    plt.figure(figsize=(10, 8))
    plt.imshow(display_grid.permute(1, 2, 0).numpy())
    plt.axis("off")
    plt.title(f"Epoch {epoch} Visualization")
    plt.show()
    plt.close()

    cycle_gan.G_src2Photo.train()
    cycle_gan.G_Photo2src.train()

# --- Helper: Save Weights ---
def save_checkpoint(cycle_gan, epoch, save_dir="./output"):
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    
    torch.save(cycle_gan.G_src2Photo.state_dict(), f"{save_dir}/G_Src2Photo_ep{epoch}.pth")
    torch.save(cycle_gan.G_Photo2src.state_dict(), f"{save_dir}/G_Photo2Src_ep{epoch}.pth")
    torch.save(cycle_gan.D_src.state_dict(), f"{save_dir}/D_Src_ep{epoch}.pth")
    torch.save(cycle_gan.D_Photo.state_dict(), f"{save_dir}/D_Photo_ep{epoch}.pth")
    print(f"--> Saved checkpoint for epoch {epoch}")


def train_cycle_gan(model, data_iter, save_freq=5, output_dir="./output"):
    epochs = model.configue["epochs"]
    device = model.configue["device"]
    
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
    
    print(f"Starting training for {epochs} epochs on {device}...")
    
    scaler = torch.cuda.amp.GradScaler()
    
    for epoch in range(epochs):
        add_metric = Accumulator(3)
        pbar = tqdm.tqdm(data_iter, total=len(data_iter), leave=False, desc=f"{epoch + 1}/{epochs}")
        
        for i, (src_img, photo_img) in enumerate(pbar):
            src_img = src_img.to(device, non_blocking=True)
            photo_img = photo_img.to(device, non_blocking=True)
            
            Frozen([model.D_src, model.D_Photo], False) 
            model.G_adam.zero_grad()

            with torch.cuda.amp.autocast():
                fake_photo = model.G_src2Photo(src_img)
                fake_src = model.G_Photo2src(photo_img)
                cycle_src = model.G_Photo2src(fake_photo)
                cycle_photo = model.G_src2Photo(fake_src)
                id_src = model.G_Photo2src(src_img)
                id_photo = model.G_src2Photo(photo_img)

                id_src_loss = model.l1loss(id_src, src_img) * model.temp * model.idt_coef
                id_photo_loss = model.l1loss(id_photo, photo_img) * model.temp * model.idt_coef
                cycle_src_loss = model.l1loss(cycle_src, src_img) * model.temp
                cycle_photo_loss = model.l1loss(cycle_photo, photo_img) * model.temp
                
                src_disc = model.D_src(fake_src)
                photo_disc = model.D_Photo(fake_photo)
                real = torch.ones(src_disc.size()).to(device)
                src_adv_loss = model.mseloss(src_disc, real)
                photo_adv_loss = model.mseloss(photo_disc, real)
                
                G_total_loss = id_src_loss + id_photo_loss + cycle_src_loss + \
                               cycle_photo_loss + src_adv_loss + photo_adv_loss

            scaler.scale(G_total_loss).backward()
            scaler.step(model.G_adam)

            Frozen([model.D_src, model.D_Photo], True)
            model.D_adam.zero_grad()

            fake_src_det = model.sample_src(fake_src.detach()).to(device)
            fake_photo_det = model.sample_photo(fake_photo.detach()).to(device)

            with torch.cuda.amp.autocast():
                src_real_disc = model.D_src(src_img)
                src_fake_disc = model.D_src(fake_src_det)
                photo_real_disc = model.D_Photo(photo_img)
                photo_fake_disc = model.D_Photo(fake_photo_det)

                real_desc = torch.full_like(src_real_disc, 0.9, device=device)
                fake_desc = torch.zeros_like(src_fake_disc, device=device)

                src_real_loss = model.mseloss(src_real_disc, real_desc)
                src_fake_loss = model.mseloss(src_fake_disc, fake_desc)
                photo_real_loss = model.mseloss(photo_real_disc, real_desc)
                photo_fake_loss = model.mseloss(photo_fake_disc, fake_desc)
                
                D_total_loss = (src_real_loss + src_fake_loss + photo_real_loss + photo_fake_loss) * 0.5

            scaler.scale(D_total_loss).backward()
            scaler.step(model.D_adam)
            scaler.update()
            
            add_metric.add(G_total_loss.item(), D_total_loss.item(), len(src_img))
            pbar.set_postfix(G_loss=f"{G_total_loss.item():.4f}", D_loss=f"{D_total_loss.item():.4f}")
        
        model.record_metric.add(add_metric[0] / add_metric[-1], add_metric[1] / add_metric[-1])
        model.G_sched.step()
        model.D_sched.step()
        
        print(f"<{epoch + 1}/{epochs}>  G_loss: {model.record_metric[0][-1]:.4f} | D_loss: {model.record_metric[1][-1]:.4f}")
        
        visualize_grid(model, data_iter, epoch + 1, device)
        
        if (epoch + 1) % save_freq == 0 or (epoch + 1) == epochs:
            # Save to output dir
            save_checkpoint(model, epoch + 1, save_dir=output_dir)
            # Also copy to /kaggle/working/checkpoints so they persist even if session crashes
            import shutil
            shutil.copy(f'{output_dir}/G_Src2Photo_ep{epoch+1}.pth', f'/kaggle/working/checkpoints/G_Src2Photo_ep{epoch+1}.pth')
            shutil.copy(f'{output_dir}/G_Photo2Src_ep{epoch+1}.pth', f'/kaggle/working/checkpoints/G_Photo2Src_ep{epoch+1}.pth')
            shutil.copy(f'{output_dir}/D_Src_ep{epoch+1}.pth',       f'/kaggle/working/checkpoints/D_Src_ep{epoch+1}.pth')
            shutil.copy(f'{output_dir}/D_Photo_ep{epoch+1}.pth',     f'/kaggle/working/checkpoints/D_Photo_ep{epoch+1}.pth')
            print(f"--> Checkpoint {epoch+1} also copied to /kaggle/working/checkpoints/")

In [ ]:
train_cycle_gan(cycle_gan, dataloader)

/tmp/ipykernel_57/1535221316.py:86: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Starting training for 50 epochs on cuda...


1/50:   0%|          | 0/625 [00:00<?, ?it/s]/tmp/ipykernel_57/1535221316.py:99: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
W0518 11:30:46.801000 57 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode
/tmp/ipykernel_57/1535221316.py:130: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
1/50:   0%|          | 1/625 [01:09<12:00:57, 69.32s/it, D_loss=4.0976, G_loss=29.4285]/tmp/ipykernel_57/1535221316.py:99: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_57/1535221316.py:130: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp

In [ ]:
# Save weights to output folder
import os
os.makedirs('./output', exist_ok=True)

torch.save(cycle_gan.G_src2Photo.state_dict(), './output/G_Src2Photo_ep5.pth')
torch.save(cycle_gan.G_Photo2src.state_dict(), './output/G_Photo2Src_ep5.pth')
torch.save(cycle_gan.D_src.state_dict(), './output/D_Src_ep5.pth')
torch.save(cycle_gan.D_Photo.state_dict(), './output/D_Photo_ep5.pth')
print("Weights saved to ./output/")

In [ ]:
import os

# Check if the folder exists yet
if os.path.exists('./output'):
    print("✅ Folder found! Files:", os.listdir('./output'))
elif os.path.exists('/kaggle/working/output'):
    print("✅ Folder found at absolute path! Files:", os.listdir('/kaggle/working/output'))
else:
    print("⏳ No folder yet. This is normal if you haven't finished Epoch 1!")

# Check the 'Working' directory generally
print("Current Working Directory Contents:", os.listdir('/kaggle/working'))

In [ ]:
import torchvision.transforms as transforms
from PIL import Image

def test_unseen_image(cycle_gan, image_path, weight_path, device):
    # 1. Load the saved weights into your Generator
    cycle_gan.G_src2Photo.load_state_dict(torch.load(weight_path))
    cycle_gan.G_src2Photo.eval() # Set to evaluation mode
    
    # 2. Prepare the image
    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
    ])
    
    # Load your test image (replace with a real path)
    img = Image.open(image_path).convert("RGB")
    input_tensor = transform(img).unsqueeze(0).to(device) 
    
    # 3. Generate the fake image
    with torch.no_grad():
        generated_tensor = cycle_gan.G_src2Photo(input_tensor)
        
    # 4. Denormalize and display
    fake_img = denormalize(generated_tensor.cpu()[0]).permute(1, 2, 0).numpy()
    real_img = denormalize(input_tensor.cpu()[0]).permute(1, 2, 0).numpy()
    
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.title("Original Test Image")
    plt.imshow(real_img)
    plt.axis("off")
    
    plt.subplot(1, 2, 2)
    plt.title("Generated Ghibli Image")
    plt.imshow(fake_img)
    plt.axis("off")
    plt.show()


In [ ]:
test_img_path = "/kaggle/input/datasets/shubham1921/real-to-ghibli-image-dataset-5k-paired-images/dataset/trainB_ghibli/studio_0026.jpg"
saved_weight = "./output/G_Src2Photo_ep5.pth" 

test_unseen_image(cycle_gan, test_img_path, saved_weight, device)

In [ ]:
test_img_path = "/kaggle/input/datasets/shubham1921/real-to-ghibli-image-dataset-5k-paired-images/dataset/trainB_ghibli/studio_0050.jpg"
saved_weight = "./output/G_Src2Photo_ep5.pth" 

test_unseen_image(cycle_gan, test_img_path, saved_weight, device)

In [ ]:
test_img_path = "/kaggle/input/datasets/shubham1921/real-to-ghibli-image-dataset-5k-paired-images/dataset/trainB_ghibli/studio_0060.jpg"
saved_weight = "./output/G_Src2Photo_ep5.pth" 

test_unseen_image(cycle_gan, test_img_path, saved_weight, device)